# Stress test on one photograph

Builds the test copies used in the paper from one non-durian orchard photograph and shows what JPEG compression adds at each size. Runs on a CPU in a few seconds.

Each copy is made from the original: reduce to *s* × *s* px (bilinear), save as JPEG at quality 70, enlarge back to 256 px (bicubic). The resize-only control skips the JPEG step.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import durian_ood as d

# Same resampling as the field cache: squash the photograph to 256 x 256 (bicubic).
x = np.asarray(Image.open(ROOT / 'examples' / 'orchard_nondurian.jpg').convert('RGB').resize((256, 256), Image.BICUBIC))

In [ ]:
levels = ['original', '224px', '160px', '112px', '80px']
rows = {'reduced + JPEG q70': levels, 'reduced only': ['original'] + [f'{s}_nojpeg' for s in levels[1:]]}

fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for r, (label, names) in enumerate(rows.items()):
    for c, name in enumerate(names):
        axes[r, c].imshow(d.apply_variant(x, name))
        axes[r, c].set_title(name.replace('_nojpeg', ''), fontsize=10)
    crop = d.apply_variant(x, names[3])[96:160, 96:160]
    axes[r, 5].imshow(Image.fromarray(crop).resize((256, 256), Image.NEAREST))
    axes[r, 5].set_title('112 px, central 64 px x4', fontsize=10)
    axes[r, 0].set_ylabel(label, fontsize=11)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

## How much does each step change the image?

Mean absolute difference from the original (grey levels, 0-255). At 112 px the JPEG step adds block artefacts on top of the loss of detail from resizing.

In [ ]:
for s in (224, 160, 112, 80):
    resized = d.apply_variant(x, f'{s}px_nojpeg').astype(float)
    compressed = d.apply_variant(x, f'{s}px').astype(float)
    print(f'{s:>3} px   resize only {np.abs(resized - x).mean():5.1f}   resize + JPEG {np.abs(compressed - x).mean():5.1f}   JPEG step alone {np.abs(compressed - resized).mean():5.1f}')

## With a trained model

Weights are not distributed (see `MODEL_CARD.md`). With your own 11-class checkpoint, the three decision rules of the paper are:

```python
import timm, torch
import torchvision.transforms.v2 as T

model = timm.create_model('efficientnet_b0.ra_in1k', num_classes=11, drop_rate=0.2)
model.load_state_dict(torch.load('best.pt', map_location='cpu')); model.eval()
tf = T.Compose([T.Resize((224, 224), antialias=True), T.ToDtype(torch.float32, scale=True),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

batch = torch.stack([tf(torch.from_numpy(d.apply_variant(x, v)).permute(2, 0, 1)) for v in levels])
with torch.no_grad():
    probs = (model(batch) / T_val).softmax(1).numpy()    # T_val from d.fit_temperature on validation logits

OTHER = 4                                             # index of 'other' in classes.json
d.accept_top1(probs, OTHER), d.accept_max_confidence(probs, OTHER), d.accept_p_other(probs, OTHER, tau)
```